# Session 5 — Quantitative representations of the data

Figures support a conclusion, so the chain of logic should be **data → interpretation → representation**, not interpretation → data → representation. This notebook is about choosing scales, showing distributions honestly, using color consistently, and preparing the extension and presentation.

**Topics** (from the Session 5 slides):
1. Review of Session 4 (team discussion).
2. Appropriate choice of scale.
3. Displaying and interpreting distributions.
4. Color across related plots.
5. Sensitivity to filtering, shown across sites.
6. Choosing an extension question and preparing the presentation.

**For each analysis, be ready to explain:** what you did, how the code works, and how you checked it.

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

### Functions
Copied from Session 4 (their contracts and tests are in Sessions 3 and 4). If you change one here, change it there too.

In [ ]:
# 3-letter amino acid code -> 1-letter code
letter = {"Ala": "A", "Arg": "R", "Asn": "N", "Asp": "D", "Cys": "C", "Gln": "Q", "Glu": "E", "Gly": "G",
          "His": "H", "Ile": "I", "Leu": "L", "Lys": "K", "Met": "M", "Phe": "F", "Pro": "P", "Ser": "S",
          "Thr": "T", "Trp": "W", "Tyr": "Y", "Val": "V"}


def read_fasta(path):
    """Return the protein sequence in a FASTA file as one string."""
    lines = path.read_text().splitlines()
    return "".join(line.strip() for line in lines if not line.startswith(">"))


def get_checked_missense(table, ref_seq):
    """Return the missense rows of table, with new columns wt, pos, and mut.

    Raises ValueError if a label cannot be parsed, is not a real substitution,
    sits outside the protein, has a WT residue that does not match ref_seq, or
    is duplicated. Missing scores are kept as NaN.
    """
    df = table[table["record_type"] == "missense"].copy()

    # "p.His75Asp" -> wt "His", site 75, mutant "Asp"
    parts = df["hgvs_pro"].str.extract(r"^p\.([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2})$")
    if parts[0].isna().any():
        raise ValueError("unparseable label(s): " + str(df.loc[parts[0].isna(), "hgvs_pro"].tolist()[:5]))
    df["wt"], df["pos"], df["mut"] = parts[0], parts[1].astype(int), parts[2]

    if not df["mut"].isin(list(letter)).all() or (df["mut"] == df["wt"]).any():
        raise ValueError("not a missense substitution")
    if not df["pos"].between(1, len(ref_seq)).all():
        raise ValueError("site outside the protein")

    # sites count from 1, so label the reference letters 1, 2, 3, ...
    ref_by_site = pd.Series(list(ref_seq), index=range(1, len(ref_seq) + 1))
    if (df["wt"].map(letter) != df["pos"].map(ref_by_site)).any():
        raise ValueError("WT residue does not match the reference")
    if df["hgvs_pro"].duplicated().any():
        raise ValueError("duplicate labels")

    df["score"] = pd.to_numeric(df["score"])
    return df[["hgvs_pro", "wt", "pos", "mut", "score", "expts"]]


def site_stats(missense, n_sites, minimum_variants=5):
    """Score statistics for each site 1..n_sites.

    n_variants counts the variants with a score (0 where there are none).
    The statistics are NaN, never 0, where n_variants < minimum_variants.
    """
    scored = missense.dropna(subset=["score"])
    stats = scored.groupby("pos")["score"].agg(["count", "mean", "median", "std", "min", "max"])
    stats = stats.reindex(range(1, n_sites + 1))
    stats["count"] = stats["count"].fillna(0).astype(int)
    stats.loc[stats["count"] < minimum_variants, ["mean", "median", "std", "min", "max"]] = np.nan
    return stats.rename(columns={"count": "n_variants"})


def make_site_region(regions, n_sites):
    """Region name for each site 1..n_sites; sites in no annotated region are "unannotated"."""
    site_region = pd.Series("unannotated", index=range(1, n_sites + 1))
    for _, r in regions.iterrows():
        site_region.loc[r["start"]:r["end"]] = r["region"]   # .loc includes both ends
    return site_region


def region_summary(missense, site_region, minimum_variants=5, minimum_experiments=1):
    """Summarize each region after filtering variants (expts) and then sites (number of variants).

    Returns (summary, included, site_table):
      summary    one row per region: included_variants, included_sites, total_sites,
                 variant_median (median of all included variants), site_median (median of site medians)
      included   the variants that passed both filters, with a region column
      site_table the included sites, with their statistics and region
    """
    kept = missense[missense["expts"] >= minimum_experiments]
    site_table = site_stats(kept, len(site_region), minimum_variants)
    site_table["region"] = site_region
    site_table = site_table[site_table["n_variants"] >= minimum_variants]

    included = kept[kept["pos"].isin(site_table.index)].copy()
    included["region"] = included["pos"].map(site_region)

    summary = pd.DataFrame({
        "included_variants": included.groupby("region").size(),
        "included_sites": site_table.groupby("region").size(),
        "total_sites": site_region.value_counts(),
        "variant_median": included.groupby("region")["score"].median(),
        "site_median": site_table.groupby("region")["median"].median(),
    }).reindex(list(site_region.unique()))
    summary[["included_variants", "included_sites"]] = summary[["included_variants", "included_sites"]].fillna(0).astype(int)
    return summary, included, site_table

Locate the data. This notebook assumes Jupyter is running with its working directory set to this notebook's folder (`code/my_code`), so the project root is two levels up. If a path check prints `False`, check `Path.cwd()`.

In [ ]:
project_root = Path.cwd().parent.parent
data_dir = project_root / "data" / "raw_data"

abundance_path = data_dir / "derived" / "pten-variant-abundance.csv"
regions_path = data_dir / "derived" / "pten-region-annotations.csv"
fasta_path = data_dir / "derived" / "pten-reference-protein.fasta"

abundance_path.exists(), regions_path.exists(), fasta_path.exists()

In [ ]:
abundance = pd.read_csv(abundance_path)
regions = pd.read_csv(regions_path)
ref_seq = read_fasta(fasta_path)
n_sites = len(ref_seq)

minimum_variants = 5
minimum_experiments = 1   # 1 = no filter on expts

missense = get_checked_missense(abundance, ref_seq)
site_region = make_site_region(regions, n_sites)
summary, included, site_table = region_summary(missense, site_region, minimum_variants, minimum_experiments)
region_order = list(site_region.unique())
summary

## 1. Review of Session 4 (team discussion)

Be ready to answer, and to describe your code clearly:
- How did you assign each PTEN variant to a region?
- How did you summarize the number of variants, included sites, and total sites in each region? How do these compare across regions?
- How did you compute the median scores (across variants, and across the median score of each included site) for each region, and how do you interpret them?
- How did you visualize and interpret score distributions across regions?
- What variables did you change to test the sensitivity of the results, and how did these changes affect the conclusions?

*Notes:*

## 2. Choosing the scale

Axis limits change visual contrast, so ask which scale is most relevant, and keep the context. Linear differences and log ratios answer different questions, so ask what the natural scale of the data is. Some missense scores are below 0, so think about what that means before taking a log. Missing data and zeros should look different.

Prompts:
- Plot the same scores with the default axis limits and with limits that show the full natural range. Which impression is the most faithful?
- Would a linear difference from WT or a log ratio fit abundance scores better? Why or why not?
- How can your plot show that a site has *no* data, as opposed to a score of zero?

In [ ]:
# Same data, two sets of axis limits

In [ ]:
# Linear scale versus a log-based scale, if it makes sense for these scores

In [ ]:
# Showing missing data differently from zeros

**Presentation notes**
- What I did:
- How the code works:
- How I checked:

## 3. Distributions

Showing only summary statistics (for example, one median per region) can hide variation. More detailed representations can show spread, range, and different sample sizes.

Prompts:
- Show the individual scores, not just the regional medians. What do you see that the medians hide?
- **Histograms** count the values in an interval, and they are sensitive to display choices: change the **bin width**, then change the **starting edge** with the same width. How many peaks would you describe in each version?
- Comparing regions: do you want to compare the *number* of positions (counts) or the *shape* within each region (proportions)? When showing several histograms together, use the same bins and report the sample sizes.
- An **empirical cumulative distribution (ECDF)** needs no bin choices. How does it compare with the histogram, and how easy is it to read?
- Does limiting the range of the data change the interpretation? Show the full range.

In [ ]:
# Individual scores next to the regional medians

In [ ]:
# Histograms: vary the bin width

In [ ]:
# Histograms: same bin width, different starting edge

In [ ]:
# Histograms by region: counts versus proportions, with shared bins

In [ ]:
# ECDF by region

**Figure checklist** (from the Session 4 slides)
- Are the labels descriptive and legible, with units where relevant?
- Is the WT reference (score = 1) visible or explained? What is "high" and what is "low"?
- Does `figsize` match the size the figure will be shown at?
- Do color and shape mean the same thing in every panel and figure?
- Is the palette right for the data: distinct colors for unordered groups, sequential for ordered amounts, diverging only around a meaningful center?
- Are colorbars labeled with visible scale marks? Do gridlines stay quiet?

**Presentation notes**
- What I did:
- How the code works:
- How I checked:

## 4. Color across related plots

When color conveys information across related plots, use a single scale and reference for consistency: the same color limits, the same center (WT = 1), and the same palette meaning in every panel.

Prompts:
- Make two related plots that use color for the abundance score. Do they share color limits? What goes wrong if they do not?

In [ ]:
# Related plots with shared color limits and one reference

**Presentation notes**
- What I did:
- How the code works:
- How I checked:

## 5. Sensitivity shown across sites

Comparing sites shows that most sites have consistent results under different filtering criteria, but not all. The slides' example filters PTEN data to require `expts >= 6`: 242 positions remain eligible, and 75 lose eligibility.

Prompts:
- Which sites stay eligible under the stricter filter, and which are lost? Where along the sequence are the lost sites?
- For the sites that stay, how much does each site's median score change? Match sites by position, not by row order.
- A site that loses eligibility has *no* new median, which is different from a change of zero.

In [ ]:
# Site medians: baseline versus expts >= 6, matched by position

In [ ]:
# Plot the comparison across sites

**Presentation notes**
- What I did:
- How the code works:
- How I checked:

## 6. Choosing an extension and preparing the presentation

Your team will define and investigate one question that extends the PTEN analyses done so far. Next class, each team presents:
- a definition of the question
- how you approached it, including **at least one explicit code example with an accompanying explanation**
- a brief summary of your results and conclusions
- plan for **5 to 7 minutes** plus brief questions. Slides and the whiteboard are both fine, and the whiteboard is encouraged for describing the problem and approach.

**Prediction sheet:**
- Define the question or goal:
- How will you approach it? Does this require new data or new analysis?
- What are your anticipated results?
- How will you ensure that your analysis is correct?

In [ ]:
# Working space for the extension

**Presentation notes**
- What I did:
- How the code works:
- How I checked:

## Reproducibility record

Run this last, after restarting the kernel and running all cells.

In [ ]:
import session_info
session_info.show(dependencies=True)